# Run settings
Do the results depend on the sampling settings (temperature, top_p, top_k) or on the number of repetitions? Each extra run has its own `run` tag; `main` is the run of the report.

In [ ]:
import sys
sys.path.insert(0, "../../..")

import matplotlib.pyplot as plt
import pandas as pd

from pipeline.analysis import stats

FIGURES = stats.RESULTS / "figures"
FIGURES.mkdir(exist_ok=True)

In [ ]:
answers = stats.readable(stats.load_answers())
answers.groupby(['model', 'run', 'temperature', 'top_p', 'top_k']).agg(
    answers=('visa_score', 'size'), mean_score=('visa_score', 'mean')).round(1)

## Gap among the 21 under each setting
Same prompts, other settings. If the gap stays about the same, the main result does not depend on the choice.

In [ ]:
df = stats.main_comparison(answers)
df = df[df['variant'] == 'base']
rows = []
for (model, run), d in df.groupby(['model', 'run']):
    spread, p = stats.permutation_p(d, stats.spread, n=1000)
    rows.append({'model': model, 'run': run, 'gap': stats.gap(stats.effects(d).loc[model]),
                 'spread': spread, 'p_spread': p})
pd.DataFrame(rows).round(3)

## How many repetitions are enough?
The gap of the main run computed from the first k repetitions only. If it settles before 5, five are enough.

In [ ]:
main = stats.main_run(df)
by_k = {(model, k): stats.gap(stats.effects(d[d['repeat'] <= k]).loc[model])
        for model, d in main.groupby('model') for k in range(1, int(main['repeat'].max()) + 1)}
pd.Series(by_k).unstack().round(2)